In [ ]:
# ICU_Project_Colab.py

# 1. IMPORTS
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 2. FILE UPLOAD
print("Please upload your ICU CSV file")
uploaded = files.upload()
filename = next(iter(uploaded.keys()))

# 3. LOAD & INITIAL INSPECTION
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')
print("Raw rows:", len(df))
print("Raw columns:", df.columns.tolist())

# 3.1 Normalize column names
df.columns = df.columns.str.strip()
print("Stripped columns:", df.columns.tolist())

# 3.2 Parse dates
df['dates'] = pd.to_datetime(df['dates'], errors='coerce')
print("Rows with valid dates:", df['dates'].notna().sum())

# 3.3 Filter valid dates only
df = df.dropna(subset=['dates'])
print("After dropping invalid dates:", len(df))

# 3.4 Rename and dedupe
df = df.rename(columns={'dayIn': 'day_in'})
before_dedupe = len(df)
df = df.drop_duplicates()
print("After deduplication:", len(df), f"(removed {before_dedupe - len(df)} duplicates)\n")

# 4. DATA ANALYSIS PARAMETERS
# 4.1 Unique counts
unique_patients = df['codes'].nunique()
unique_stays = df[['codes', 'id']].drop_duplicates().shape[0]

# 4.2 Arrival rate (λ)
arrivals = df[df['day_in'] == 1].groupby(df['dates'].dt.date).size()
lambda_est = arrivals.mean()

# 4.3 Length of Stay (LOS)
los_df = df.groupby(['codes', 'id'])['day_in'].max().reset_index(name='length_of_stay')
los = los_df['length_of_stay'].dropna()
mean_los = los.mean()
shape, loc, scale = stats.lognorm.fit(los, floc=0)

# 4.4 Return rate
return_rate = (los_df.groupby('codes')['id'].nunique() > 1).mean()

# 5. FIGURE 1: Daily ICU Admissions with Poisson Fit
x = np.arange(0, arrivals.max() + 1)
plt.figure()
plt.hist(arrivals, bins=range(0, arrivals.max()+2), density=True, alpha=0.6)
plt.plot(x, stats.poisson.pmf(x, lambda_est), marker='o')
plt.title('Figure 1: Daily ICU Admissions with Poisson Fit')
plt.xlabel('Admissions per Day')
plt.ylabel('Probability')
plt.show()

# 6. FIGURE 2: LOS Distribution with Lognormal Fit
x_los = np.linspace(1, los.max(), 100)
plt.figure()
plt.hist(los, bins=20, density=True, alpha=0.6)
plt.plot(x_los, stats.lognorm.pdf(x_los, shape, loc, scale))
plt.title('Figure 2: ICU Length of Stay Distribution with Lognormal Fit')
plt.xlabel('Length of Stay (days)')
plt.ylabel('Density')
plt.show()

# 7. SIMULATION (Hourly, 365 days)
SIM_DAYS = 365
CAPACITY = 15
ARR_HR = lambda_est / 24
beds, util, rej_list = [], [], []
rej = 0

for _ in range(SIM_DAYS):
    for _ in range(24):
        beds = [b-1 for b in beds if b-1 > 0]
        for __ in range(np.random.poisson(ARR_HR)):
            if len(beds) < CAPACITY:
                beds.append(int(np.random.lognormal(shape, scale) * 24))
            else:
                rej += 1
        util.append(len(beds) / CAPACITY)
        rej_list.append(rej)

# 8. FIGURE 3: Utilization Over Time
plt.figure()
plt.plot(util)
plt.title('Figure 3: ICU Utilization Over Time (Hourly)')
plt.xlabel('Hour')
plt.ylabel('Utilization Rate')
plt.show()

# 9. FIGURE 4: Cumulative Rejections
plt.figure()
plt.plot(rej_list)
plt.title('Figure 4: Cumulative Rejected Admissions Over Time (Hourly)')
plt.xlabel('Hour')
plt.ylabel('Cumulative Rejections')
plt.show()

# 10. PREDICTIVE MODELING SETUP
df['target'] = np.where(
    ((df['bad'] == 1) & (df['last'] == 'last')) |
    ((df['bad'] == 1) & (df['last'] == 'in')),
    1, 0
)

features = ['age', 'gender', 'SAPS', 'Glasgow', 'TISS', 'PA', 'FC', 'Temp']
X = df[features]
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# 11. PIPELINE: Imputation & Scaling
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train_proc = pipe.fit_transform(X_train)
X_test_proc = pipe.transform(X_test)

# 12. EVALUATE MODELS
def evaluate_model(model, X_tr, y_tr, X_te, y_te, threshold=0.45):
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_te)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_te, preds),
        'precision': precision_score(y_te, preds),
        'recall': recall_score(y_te, preds),
        'f1': f1_score(y_te, preds),
        'roc_auc': roc_auc_score(y_te, probs),
        'conf_matrix': confusion_matrix(y_te, preds)
    }

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(probability=True),
    'Decision Tree': DecisionTreeClassifier(),
    'K-NN': KNeighborsClassifier()
}

for name, mdl in models.items():
    res = evaluate_model(mdl, X_train_proc, y_train, X_test_proc, y_test)
    print(f"{name}: Acc={res['accuracy']:.2f}, Prec={res['precision']:.2f}, Rec={res['recall']:.2f}, F1={res['f1']:.2f}, AUC={res['roc_auc']:.2f}")
    print("Confusion Matrix:\n", res['conf_matrix'], "\n")

# 13. ROC CURVES
plt.figure()
for name, mdl in models.items():
    mdl.fit(X_train_proc, y_train)
    probs = mdl.predict_proba(X_test_proc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs):.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Discharge Models')
plt.legend()
plt.show()

# 14. FORMULAS & SUMMARY TABLES
print("\n=== Formulas ===")
print("Arrival rate (λ) = Total Admissions / Total Days")
print("Mean LOS = sum(LOS_i) / N")
print("Return rate = (Patients with >1 Stay / Total Unique Patients) * 100%")
print("Average Utilization = Occupied Bed-Hours / (Capacity * Total Hours)")
print("Total Rejections = sum(arrivals when full capacity)\n")

data_summary = {
    'Metric': [
        'Total rows after cleaning',
        'Unique patients',
        'Unique stays',
        'Arrival rate (λ)',
        'Mean LOS',
        'Log-normal fit (shape, scale)',
        'Return rate'
    ],
    'Value': [
        f"{len(df):,}",
        f"{unique_patients}",
        f"{unique_stays}",
        f"{lambda_est:.2f} per day",
        f"{mean_los:.2f} days",
        f"shape={shape:.2f}, scale={scale:.2f}",
        f"{return_rate*100:.1f}%"
    ]
}
print("=== Data Cleaning & Analysis Summary ===")
print(pd.DataFrame(data_summary).to_string(index=False))

avg_util = sum(util) / (SIM_DAYS * 24)
sim_summary = {
    'KPI': [
        'Average bed utilization',
        'Total patient rejections (year)'
    ],
    'Result': [
        f"{avg_util*100:.1f}%",
        f"{rej}"
    ]
}
print("\n=== Simulation Outcomes Summary ===")
print(pd.DataFrame(sim_summary).to_string(index=False))


In [ ]:
# ICU_Project_Colab.py

# 1. IMPORTS
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 2. FILE UPLOAD
print("Please upload your ICU CSV file")
uploaded = files.upload()
filename = next(iter(uploaded.keys()))

# 3. LOAD & CLEAN DATA
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')
df.columns = df.columns.str.strip()
df.rename(columns={'dayIn': 'day_in'}, inplace=True)

# 3.1 Parse dates (retain all rows)
df['dates'] = pd.to_datetime(df['dates'], errors='coerce')
print("Rows with valid dates for arrivals:", df['dates'].notna().sum(), "of", len(df))
print("Total rows after cleaning:", len(df))

# 4. DATA ANALYSIS PARAMETERS
unique_patients = df['codes'].nunique()
los_df = df.groupby(['codes', 'id'])['day_in'].max().reset_index(name='length_of_stay')
unique_stays = los_df.shape[0]

# 4.1 Arrival rate (λ) from valid admission dates only
arrivals = df[(df['day_in'] == 1) & df['dates'].notna()] \
             .groupby(df['dates'].dt.date).size()
lambda_est = arrivals.mean()

# 4.2 Length of Stay (LOS)
los = los_df['length_of_stay'].dropna()
mean_los = los.mean()
shape, loc, scale = stats.lognorm.fit(los, floc=0)

# 4.3 Return rate
return_rate = (los_df.groupby('codes')['id'].nunique() > 1).mean()

# 5. FIGURE 1: Daily ICU Admissions with Poisson Fit
x = np.arange(0, arrivals.max() + 1)
plt.figure()
plt.hist(arrivals, bins=range(0, arrivals.max()+2), density=True, alpha=0.6)
plt.plot(x, stats.poisson.pmf(x, lambda_est), marker='o')
plt.title('Figure 1: Daily ICU Admissions with Poisson Fit')
plt.xlabel('Admissions per Day'); plt.ylabel('Probability')
plt.show()

# 6. FIGURE 2: LOS Distribution with Lognormal Fit
x_los = np.linspace(1, los.max(), 100)
plt.figure()
plt.hist(los, bins=20, density=True, alpha=0.6)
plt.plot(x_los, stats.lognorm.pdf(x_los, shape, loc, scale))
plt.title('Figure 2: ICU Length of Stay Distribution with Lognormal Fit')
plt.xlabel('Length of Stay (days)'); plt.ylabel('Density')
plt.show()

# 7. SIMULATION (Hourly, 365 days)
SIM_DAYS, CAPACITY = 365, 15
ARR_HR = lambda_est / 24
beds, util, rej_list = [], [], []
rej = 0

for _ in range(SIM_DAYS):
    for _ in range(24):
        beds = [b-1 for b in beds if b-1 > 0]
        for __ in range(np.random.poisson(ARR_HR)):
            if len(beds) < CAPACITY:
                beds.append(int(np.random.lognormal(shape, scale) * 24))
            else:
                rej += 1
        util.append(len(beds) / CAPACITY)
        rej_list.append(rej)

# 8. FIGURE 3: Utilization Over Time
plt.figure()
plt.plot(util)
plt.title('Figure 3: ICU Utilization Over Time (Hourly)')
plt.xlabel('Hour'); plt.ylabel('Utilization Rate')
plt.show()

# 9. FIGURE 4: Cumulative Rejections
plt.figure()
plt.plot(rej_list)
plt.title('Figure 4: Cumulative Rejected Admissions Over Time (Hourly)')
plt.xlabel('Hour'); plt.ylabel('Cumulative Rejections')
plt.show()

# 10. PREDICTIVE MODELING SETUP
df['target'] = np.where(
    ((df['bad'] == 1) & (df['last'] == 'last')) |
    ((df['bad'] == 1) & (df['last'] == 'in')),
    1, 0
)
features = ['age', 'gender', 'SAPS', 'Glasgow', 'TISS', 'PA', 'FC', 'Temp']
X, y = df[features], df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# 11. PIPELINE: Imputation & Scaling
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train_proc = pipe.fit_transform(X_train)
X_test_proc = pipe.transform(X_test)

# 12. EVALUATE MODELS
def evaluate_model(model, X_tr, y_tr, X_te, y_te, threshold=0.45):
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_te)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_te, preds),
        'precision': precision_score(y_te, preds),
        'recall': recall_score(y_te, preds),
        'f1': f1_score(y_te, preds),
        'roc_auc': roc_auc_score(y_te, probs),
        'conf_matrix': confusion_matrix(y_te, preds)
    }

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(probability=True),
    'Decision Tree': DecisionTreeClassifier(),
    'K-NN': KNeighborsClassifier()
}
for name, mdl in models.items():
    res = evaluate_model(mdl, X_train_proc, y_train, X_test_proc, y_test)
    print(f"{name}: Acc={res['accuracy']:.2f}, Prec={res['precision']:.2f}, Rec={res['recall']:.2f}, F1={res['f1']:.2f}, AUC={res['roc_auc']:.2f}")
    print("Confusion Matrix:\n", res['conf_matrix'], "\n")

# 13. ROC CURVES
plt.figure()
for name, mdl in models.items():
    mdl.fit(X_train_proc, y_train)
    probs = mdl.predict_proba(X_test_proc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs):.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Discharge Models'); plt.legend(); plt.show()

# 14. FORMULAS & SUMMARY TABLES
print("\n=== Formulas ===")
print("Arrival rate (λ) = Total Admissions / Total Days")
print("Mean LOS = sum(LOS_i) / N")
print("Return rate = (Patients with >1 Stay / Total Unique Patients) * 100%")
print("Average Utilization = Occupied Bed-Hours / (Capacity * Total Hours)")
print("Total Rejections = sum(arrivals when full capacity)\n")

data_summary = {
    'Metric': [
        'Total rows after cleaning',
        'Unique patients',
        'Unique stays',
        'Arrival rate (λ)',
        'Mean LOS',
        'Log-normal fit (shape, scale)',
        'Return rate'
    ],
    'Value': [
        f"{len(df):,}",
        f"{unique_patients}",
        f"{unique_stays}",
        f"{lambda_est:.2f} per day",
        f"{mean_los:.2f} days",
        f"shape={shape:.2f}, scale={scale:.2f}",
        f"{return_rate*100:.1f}%"
    ]
}
print("=== Data Cleaning & Analysis Summary ===")
print(pd.DataFrame(data_summary).to_string(index=False))

avg_util = np.mean(util)
sim_summary = {
    'KPI': [
        'Average bed utilization',
        'Total patient rejections (year)'
    ],
    'Result': [
        f"{avg_util*100:.1f}%",
        f"{rej}"
    ]
}
print("\n=== Simulation Outcomes Summary ===")
print(pd.DataFrame(sim_summary).to_string(index=False))
